# 14 — FlashAttention Concepts with Online Softmax

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Standard attention materializes a large `T × T` score matrix. FlashAttention computes exact attention in tiles and avoids writing the full attention matrix to high-bandwidth memory.

This notebook implements the core **online softmax** idea on CPU. It is not a CUDA kernel, but it teaches the algorithmic reason FlashAttention is memory-efficient.

In [ ]:
def normal_attention(q, k, v):
    scores = q @ k.T / math.sqrt(q.size(-1))
    probs = torch.softmax(scores, dim=-1)
    return probs @ v

# The next cell defines an online tiled implementation. This cell keeps the standard baseline only.


## Online tiled attention

The implementation below keeps running row-wise max and denominator statistics so it can combine tiles without materializing the full attention matrix.

In [ ]:
def online_tiled_attention(q, k, v, block_size=4):
    T,D = q.shape
    scale = 1 / math.sqrt(D)
    m = torch.full((T,), -float('inf'), device=q.device)
    l = torch.zeros(T, device=q.device)
    out = torch.zeros(T, v.size(-1), device=q.device)
    for start in range(0, T, block_size):
        end = min(T, start + block_size)
        scores = (q @ k[start:end].T) * scale
        m_new = torch.maximum(m, scores.max(dim=-1).values)
        alpha = torch.exp(m - m_new)
        p = torch.exp(scores - m_new[:, None])
        l_new = l * alpha + p.sum(dim=-1)
        out = out * ((l * alpha) / (l_new + 1e-9))[:, None] + (p @ v[start:end]) / (l_new[:, None] + 1e-9)
        m, l = m_new, l_new
    return out

q = torch.randn(12, 16)
k = torch.randn(12, 16)
v = torch.randn(12, 16)
a = normal_attention(q,k,v)
b = online_tiled_attention(q,k,v,block_size=3)
print('max diff:', float((a-b).abs().max()))


In [ ]:
def attention_memory_estimate(T, dtype_bytes=2):
    return T*T*dtype_bytes/1e9
pd.DataFrame([{'T':T, 'score_matrix_GB_per_head':attention_memory_estimate(T)} for T in [4096, 8192, 32768, 131072]])


## Production notes

FlashAttention is about IO awareness, not approximate attention. In production you normally use optimized kernels through PyTorch, xFormers, FlashAttention, vLLM, TensorRT-LLM, or your serving runtime—not this notebook implementation.